<a href="https://colab.research.google.com/github/silvatria/pembelajaran-mesin-254107023001/blob/main/JS06_TL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Bagian 1: Klasifikasi Dataset Voice (voice.csv) dengan SVM
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# 1. Load dataset
df = pd.read_csv('voice.csv')
X = df.drop('label', axis=1)
y = df['label']

# 2. Standarisasi fitur (sangat direkomendasikan untuk SVM)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Pengujian kombinasi split dan kernel
splits = {'70:30': 0.3, '80:20': 0.2}
kernels = ['linear', 'poly', 'rbf']
results = []

for split_name, test_size in splits.items():
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=test_size, random_state=42)
    for kernel in kernels:
        model = SVC(kernel=kernel, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results.append({
            'Rasio Split (Train:Test)': '70:30' if test_size==0.3 else '80:20',
            'Kernel': kernel.upper(),
            'Akurasi': round(acc * 100, 2)
        })

# 4. Tampilkan Tabulasi Performa
tabulasi_df = pd.DataFrame(results)
print(tabulasi_df.to_string(index=False))

'''
(Klasifikasi Dataset Voice): Eksperimen ini membandingkan performa model Support Vector Machine (SVM)
menggunakan variasi kernel (linear, polynomial, dan rbf) serta rasio pembagian data train-test (70:30 dan 80:20)
pada dataset akustik suara, di mana kernel RBF dan rasio 80:20 secara konsisten menghasilkan tingkat akurasi
tertinggi mencapai di atas 98%.
'''

Rasio Split (Train:Test) Kernel  Akurasi
                   70:30 LINEAR    97.06
                   70:30   POLY    95.69
                   70:30    RBF    98.11
                   80:20 LINEAR    97.63
                   80:20   POLY    96.85
                   80:20    RBF    98.26


'\nPraktikum 1 (Klasifikasi Dataset Voice): Eksperimen ini membandingkan performa model Support Vector Machine (SVM) \nmenggunakan variasi kernel (linear, polynomial, dan rbf) serta rasio pembagian data train-test (70:30 dan 80:20) \npada dataset akustik suara, di mana kernel RBF dan rasio 80:20 secara konsisten menghasilkan tingkat akurasi \ntertinggi mencapai di atas 98%.\n'

In [ ]:
from pathlib import Path
import matplotlib.image as mpimg
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# 1. Fungsi load dataset yang otomatis masuk ke subfolder day/night
def load_dataset(img_dir):
    p = Path(img_dir)
    img_list = []
    for dir_path in p.iterdir():
        if dir_path.is_dir():
            label = dir_path.name
            for file in dir_path.glob('*.jpg'):
                img = mpimg.imread(file)
                if img is not None:
                    img_list.append((img, label))
    return img_list

# Arahkan ke folder training yang bener sesuai struktur
train_dir = 'images/training'
train_img = load_dataset(train_dir)

# 2. Ekstraksi fitur histogram dari train_img
X_features = []
y_labels = []

for img, label in train_img:
    hist_r, _ = np.histogram(img[:, :, 0], bins=8, range=(0, 256))
    hist_g, _ = np.histogram(img[:, :, 1], bins=8, range=(0, 256))
    hist_b, _ = np.histogram(img[:, :, 2], bins=8, range=(0, 256))

    hist_features = np.concatenate([hist_r, hist_g, hist_b])
    X_features.append(hist_features)
    y_labels.append(label)

X_features = np.array(X_features)
y = np.array(y_labels)

# 3. Split data 80:20
X_train, X_test, y_train, y_test = train_test_split(X_features, y, test_size=0.2, random_state=42)

# 4. Hyperparameter Tuning untuk Kernel RBF
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1]
}

grid_search = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# 5. Evaluasi Hasil
best_model = grid_search.best_estimator_
print(f"Hyperparameter Terbaik: {grid_search.best_params_}")

y_pred = best_model.predict(X_test)
acc_rbf_img = accuracy_score(y_test, y_pred)

print(f"Akurasi SVM RBF (Citra Siang/Malam): {acc_rbf_img * 100:.2f}%")
print(classification_report(y_test, y_pred))

'''
Praktikum ini menerapkan model SVM dengan kernel RBF untuk mengklasifikasikan citra kondisi siang dan malam hari
berdasarkan ekstraksi fitur histogram warna RGB dari direktori data latih, yang kemudian dioptimasi menggunakan
hyperparameter tuning guna mendapatkan performa akurasi terbaik pada rasio split 80:20.
'''

Hyperparameter Terbaik: {'C': 10, 'gamma': 'scale'}
Akurasi SVM RBF (Citra Siang/Malam): 100.00%
              precision    recall  f1-score   support

         day       1.00      1.00      1.00        24
       night       1.00      1.00      1.00        24

    accuracy                           1.00        48
   macro avg       1.00      1.00      1.00        48
weighted avg       1.00      1.00      1.00        48



'\nPraktikum ini menerapkan model SVM dengan kernel RBF untuk mengklasifikasikan citra kondisi siang dan malam hari\nberdasarkan ekstraksi fitur histogram warna RGB dari direktori data latih, yang kemudian dioptimasi menggunakan \nhyperparameter tuning guna mendapatkan performa akurasi terbaik pada rasio split 80:20.\n'